# PGA Tour – Traditionell statistikanalys
Spelare × säsong, 2010–2018. Källa: PGA Tour via Kaggle (jmpark746).

In [67]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [68]:
csv_path = "data/pga-traditional/pgaTourData.csv"

df = pd.read_csv(csv_path)

df["Player Name"] = df["Player Name"].astype("category")
df["Money"] = df["Money"].str.replace(r"[$,]", "", regex=True).astype(float)

In [73]:
df.head()

,Player Name,Rounds,Fairway Percentage,Year,Avg Distance,gir,Average Putts,Average Scrambling,Average Score,Points,Average SG Putts,Average SG Total,SG:OTT,SG:APR,SG:ARG
0,Henrik Stenson,60.0,75.19,2018,291.5,73.51,29.93,60.67,69.617,868.0,-0.207,1.153,0.427,0.960,-0.027
1,Ryan Armour,109.0,73.58,2018,283.5,68.22,29.31,60.13,70.758,1006.0,-0.058,0.337,-0.012,0.213,0.194
2,Chez Reavie,93.0,72.24,2018,286.5,68.67,29.12,62.27,70.432,1020.0,0.192,0.674,0.183,0.437,-0.137
3,Ryan Moore,78.0,71.94,2018,289.2,68.80,29.17,64.16,70.015,795.0,-0.271,0.941,0.406,0.532,0.273
4,Brian Stuard,103.0,71.44,2018,278.9,67.12,29.11,59.23,71.038,421.0,0.164,0.062,-0.227,0.099,0.026


In [72]:
df.describe()

,Rounds,Fairway Percentage,Year,Avg Distance,gir,Average Putts,Average Scrambling,Average Score,Points,Average SG Putts,Average SG Total,SG:OTT,SG:APR,SG:ARG
count,1678.000000,1678.000000,1678.000000,1678.000000,1678.000000,1678.000000,1678.000000,1678.000000,1674.000000,1678.000000,1678.000000,1678.000000,1678.000000,1678.000000
mean,78.711561,61.440560,2014.004768,290.807688,65.661675,29.163331,58.115638,70.921961,631.125448,0.025641,0.148105,0.037759,0.065015,0.019974
std,14.274137,5.058845,2.608637,8.916631,2.745411,0.518468,3.384769,0.698305,452.741472,0.343787,0.694923,0.379892,0.380952,0.223361
min,45.000000,43.020000,2010.000000,266.400000,53.540000,27.510000,44.010000,68.698000,3.000000,-1.475000,-3.209000,-1.717000,-1.680000,-0.930000
25%,69.000000,57.942500,2012.000000,284.900000,63.830000,28.810000,55.900000,70.494250,322.000000,-0.187000,-0.254750,-0.190250,-0.180750,-0.123000
50%,79.500000,61.430000,2014.000000,290.550000,65.790000,29.140000,58.275000,70.902000,530.000000,0.040000,0.147000,0.056000,0.081000,0.022000
75%,89.000000,64.910000,2016.000000,296.400000,67.580000,29.520000,60.420000,71.342750,813.750000,0.257000,0.568500,0.291500,0.314500,0.175000
max,120.000000,76.880000,2018.000000,319.700000,73.520000,31.000000,69.330000,74.400000,4169.000000,1.130000,2.406000,1.485000,1.533000,0.660000


## Vad krävs det för att spela på PGA Tour?

De flesta golfare ser höjdpunktsklipp av spektakulära slag. Det skapar en skev bild av vad som faktiskt skiljer en Tour-spelare från en amatör. Det här notebooket utgår från verklig statistik — driving, greener, puttning, scrambling — för att besvara en konkret fråga:

> *Vad är det egentligen som skiljer PGA Tour-proffsen från resten?*

**Hypotes:** Proffsen vinner inte sina slag från tee-boxen. De vinner dem genom att konsekvent lägga sig i bra position — på greenen, nära hål — och sedan konvertera. En amatör som eliminerar tre-puttningar och höjer sin GIR% mot tour-snittet sparar fler slag än vad ett extra drivavstånd ger.

---

### Datarensning

Datasetet innehåller 2 312 rader men 634 (27%) saknar all statistik — spelare som inte spelade tillräckligt många ronder under säsongen för att kvalificera för officiell statistik. Dessa rader tas bort.

**Kolumner som tas bort:**
- `Money` — priset i dollar, inte relevant för prestationsanalysen
- `Wins` och `Top 10` — utfallsvariabler som mäter resultat, inte hur spelaren spelar; kan tas in igen om vi vill analysera vad som separerar vinnare från resten
- `Points` konverteras från sträng till numeriskt värde (tusenavgränsare i källdata)

**Strokes Gained-kolumner** (`Average SG Putts`, `SG:OTT`, `SG:APR`, `SG:ARG`) behålls men används inte i huvudanalysen — de är för komplexa för direktjämförelse med amatörer men fungerar som verifiering av hypotesen i slutet av rapporten.

In [69]:
df = df.dropna(subset=["Average Score"]).copy()

df = df.drop(columns=["Money", "Wins", "Top 10"])
df["Points"] = pd.to_numeric(df["Points"].str.replace(",", ""), errors="coerce")

print(f"Rader efter rensning: {df.shape[0]}")
print(f"Kolumner: {df.shape[1]}")
print(df.isna().sum()[df.isna().sum() > 0])

Rader efter rensning: 1678
Kolumner: 15
Points    4
dtype: int64
